# 02 — Clustering Kolektor (segmentasi perilaku belanja)

> **Data SINTETIS.** Segmen di notebook ini menggambarkan struktur yang ditanam di `src/generate.py`, bukan perilaku kolektor nyata.

**Tujuan:** mengelompokkan 200 kolektor menjadi segmen berdasarkan *cara mereka belanja* (anggaran, frekuensi, keterpusatan selera, minat pada seniman besar/tren),
untuk dashboard seniman ("Siapa Pembelimu") dan label segmen di rekomendasi.

**Aturan main (ditetapkan sebelum melihat hasil):**
1. Fitur klaster = perilaku, bukan selera aliran (alasan & bukti di bagian 2).
2. K dipilih **tanpa label**: di antara K yang stabil (ARI bootstrap ≥ 0,70), ambil BIC GMM terendah.
3. Label asli generator (`kolektor_label_asli.csv`) **hanya untuk validasi setelah K final** (bagian 5) — tidak ikut memilih apa pun.
4. Nama segmen berasal dari karakteristik centroid, bukan dari label generator.

Logika inti ada di `src/clustering.py` (dipakai juga oleh `scripts/latih_clustering.py` dan `scripts/hitung_segmen.py`). Artefak & laporan resmi dibuat oleh
`python scripts/latih_clustering.py` → `models/clustering_kolektor.joblib`, `models/LAPORAN_CLUSTERING.md`.

In [ ]:
%matplotlib inline
import os, sys, warnings
from pathlib import Path

_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "data" / "training").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml-recommender/ (berisi src/ & data/training/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler

from src import clustering as C

pd.set_option("display.width", 200)
print("folder kerja:", Path.cwd())

## 1. Data
`data/training/clustering_kolektor.csv` — 1 baris per kolektor (snapshot akhir periode). Arti kolom: `data/training/KAMUS_DATA.md`.

In [ ]:
df = pd.read_csv("data/training/clustering_kolektor.csv")
print(df.shape, "| NaN:", int(df.isna().sum().sum()))
print("\nkolom fitur kandidat:", [c for c in df.columns if c != "kolektor_id"])

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].hist(np.log10(df["harga_rata2_idr"]), bins=25, color="tab:blue"); ax[0].set_title("log10(harga rata-rata Rp) — condong kanan")
ax[1].hist(df["n_pembelian"], bins=25, color="tab:orange"); ax[1].set_title("jumlah pembelian per kolektor")
ax[2].hist(df["porsi_gaya_teratas"], bins=20, color="tab:green"); ax[2].set_title("porsi aliran favorit (1 = setia 1 aliran)")
plt.tight_layout(); plt.show()

## 2. Keputusan fitur: perilaku vs ikut menyertakan porsi aliran
Kumpulan **A** = semua 30 fitur. Kumpulan **B** = 19 fitur perilaku (tanpa 11 kolom `porsi_gaya_<aliran>`).
Jika porsi aliran ikut dipakai, klaster cenderung terbentuk menurut "penggemar aliran X", bukan menurut perilaku belanja. Kita uji **tanpa label**: silhouette, stabilitas bootstrap.

In [ ]:
hasil = {}
for nama, termasuk in (("A: semua fitur", True), ("B: perilaku (tanpa porsi aliran)", False)):
    fitur = C.kolom_fitur(df, termasuk_porsi_aliran=termasuk)
    z = StandardScaler().fit_transform(C.matriks(df, fitur, C.kolom_log(fitur)))
    hasil[nama] = (len(fitur), C.evaluasi_k(z))
    print(f"{nama}: {len(fitur)} fitur — selesai")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for nama, (n, t) in hasil.items():
    ax[0].plot(t["k"], t["silhouette"], "o-", label=f"{nama} ({n})")
    ax[1].errorbar(t["k"], t["stabilitas_bootstrap"], yerr=t["stabilitas_sd"], fmt="o-", capsize=3, label=f"{nama} ({n})")
ax[0].set_title("Silhouette (makin tinggi makin baik)"); ax[0].set_xlabel("K"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].axhline(C.STABIL_MIN, color="gray", ls="--", lw=1, label=f"ambang {C.STABIL_MIN}")
ax[1].set_title("Stabilitas bootstrap (ARI)"); ax[1].set_xlabel("K"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print("Kumpulan B lebih baik di hampir semua K -> dipakai.")

## 3. Memilih K
Aturan: di antara K dengan stabilitas ≥ 0,70, pilih **BIC GMM terendah**. (Silhouette selalu turun saat K naik, jadi tidak bisa jadi satu-satunya dasar.)

In [ ]:
tabel_b = hasil["B: perilaku (tanpa porsi aliran)"][1]
k_terpilih = C.pilih_k(tabel_b)
tampil = tabel_b.copy()
tampil["layak"] = np.where(tampil["stabilitas_bootstrap"] >= C.STABIL_MIN, "ya", "tidak")
tampil["terpilih"] = np.where(tampil["k"] == k_terpilih, "<--", "")
print(tampil.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(tabel_b["k"], tabel_b["silhouette"], "o-"); ax[0].set_title("Silhouette")
ax[1].errorbar(tabel_b["k"], tabel_b["stabilitas_bootstrap"], yerr=tabel_b["stabilitas_sd"], fmt="o-", capsize=3)
ax[1].axhline(C.STABIL_MIN, color="gray", ls="--"); ax[1].set_title("Stabilitas bootstrap ± sd")
ax[2].plot(tabel_b["k"], tabel_b["bic_gmm"], "o-"); ax[2].set_title("BIC GMM (makin rendah makin baik)")
for a in ax: a.axvline(k_terpilih, color="tab:red", alpha=.4); a.set_xlabel("K"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"K terpilih = {k_terpilih}")
print("CATATAN: stabilitas K terpilih sangat dekat ambang dan sd-nya besar -> jumlah segmen adalah perkiraan, bukan angka pasti.")

## 4. Model akhir & profil segmen
K-Means pada fitur B (log1p → StandardScaler). Nama & deskripsi dibuat dari centroid (`beri_nama`).

In [ ]:
bundle = C.latih(df, k=k_terpilih, tabel_k=tabel_b)
asign = C.tetapkan(bundle, df)

prof = pd.DataFrame({sid: {"nama": s["nama"], "ukuran": s["ukuran"], **{k: v for k, v in s["profil"].items()}, "aliran_favorit": s["aliran_favorit"]}
                     for sid, s in sorted(bundle["segmen"].items())}).T
print(prof[["nama", "ukuran", "n_pembelian", "harga_rata2_idr", "frekuensi_beli_per_bulan", "porsi_gaya_teratas",
            "porsi_beli_seniman_mapan", "porsi_beli_seniman_pemula", "aliran_favorit"]].to_string())
print()
for sid, s in sorted(bundle["segmen"].items()):
    print(f"{sid}. {s['nama']}: {s['deskripsi']}")

In [ ]:
# heatmap: seberapa jauh profil tiap segmen dari median populasi (dalam simpangan baku antar-segmen)
kol = C.KOLOM_PROFIL
M = pd.DataFrame({f"{sid}. {s['nama']}": s["profil"] for sid, s in sorted(bundle["segmen"].items())}).T[kol].astype(float)
Zs = (M - M.mean()) / M.std(ddof=0).replace(0, 1)
fig, ax = plt.subplots(figsize=(11, 3.6))
im = ax.imshow(Zs.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(kol)), kol, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(len(Zs)), Zs.index, fontsize=9)
for i in range(Zs.shape[0]):
    for j in range(Zs.shape[1]): ax.text(j, i, f"{Zs.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, label="z-score antar segmen"); ax.set_title("Profil segmen (merah = tinggi, biru = rendah)")
plt.tight_layout(); plt.show()

# proyeksi 2D hanya untuk melihat (bukan dasar keputusan)
Z = bundle["scaler"].transform(C.matriks(df, bundle["fitur"], bundle["kolom_log"]))
xy = PCA(2, random_state=0).fit_transform(Z)
fig, ax = plt.subplots(figsize=(7, 5))
for sid, s in sorted(bundle["segmen"].items()):
    m = asign["segmen_id"].to_numpy() == sid
    ax.scatter(xy[m, 0], xy[m, 1], s=22, alpha=.8, label=f"{sid}. {s['nama']}")
ax.set_title("Kolektor pada 2 komponen utama (batas antar segmen memang kabur)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Validasi terhadap label asli generator (HANYA setelah K final)
Di dunia nyata label ini tidak ada. Di sini dipakai untuk mengukur seberapa jauh struktur yang kita tanam terbaca oleh clustering.
**Jangan** dipakai untuk mengubah K, fitur, atau nama segmen.

In [ ]:
lab = pd.read_csv("data/training/kolektor_label_asli.csv")
m = asign.merge(lab, on="kolektor_id")
ari = adjusted_rand_score(m["segmen_asli"], m["segmen_id"]); nmi = normalized_mutual_info_score(m["segmen_asli"], m["segmen_id"])
print(f"ARI = {ari:.3f} | NMI = {nmi:.3f}")
cross = pd.crosstab(m["segmen_id"].map(lambda s: f"{s}. {bundle['segmen'][s]['nama']}"), m["segmen_asli"])
print(cross.to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
P = cross.div(cross.sum(axis=0), axis=1)
im = ax.imshow(P.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(P.shape[1]), P.columns, rotation=30, ha="right"); ax.set_yticks(range(P.shape[0]), P.index)
for i in range(P.shape[0]):
    for j in range(P.shape[1]): ax.text(j, i, f"{cross.iloc[i, j]}", ha="center", va="center", fontsize=9, color="white" if P.iloc[i, j] > .5 else "black")
fig.colorbar(im, ax=ax, label="porsi dari segmen asli"); ax.set_title("Segmen hasil clustering (baris) vs segmen asli generator (kolom)")
plt.tight_layout(); plt.show()

## 6. Konsistensi dengan artefak resmi
Artefak resmi dibuat oleh `scripts/latih_clustering.py` (jangan ditimpa dari notebook). Sel ini hanya memastikan hasil notebook sama dengan artefak yang dipakai `scripts/hitung_segmen.py`.

In [ ]:
resmi = joblib.load("models/clustering_kolektor.joblib")
a2 = C.tetapkan(resmi, df)
print(f"artefak resmi: {resmi['versi']} | K = {resmi['k']} | dilatih {resmi['dilatih_pada']}")
cocok = adjusted_rand_score(asign["segmen_id"], a2["segmen_id"])
print(f"kesamaan penugasan notebook vs artefak resmi (ARI): {cocok:.3f}  ->", "SAMA" if cocok == 1.0 else "BEDA — jalankan ulang scripts/latih_clustering.py")
assert (asign["segmen_nama"].to_numpy() == a2["segmen_nama"].to_numpy()).all(), "nama segmen berbeda dari artefak resmi"

## 7. Catatan
* **Keterbatasan:** data sintetis, 200 kolektor, median ±6 pembelian per kolektor → fitur berisik, batas segmen kabur (silhouette ±0,18), keputusan K berada di tepi ambang stabilitas. Lihat `models/LAPORAN_CLUSTERING.md` bagian 5.
* **Jangan dipakai sebagai fitur klasifikasi:** segmen dihitung dari seluruh riwayat sampai akhir periode → membocorkan masa depan.
* **Pemakaian:** `python scripts/latih_clustering.py` (latih + laporan) lalu `python scripts/hitung_segmen.py` (isi tabel `kolektor_segmen` di Neon; dibaca dashboard seniman).

## 8. Evaluasi Model — rapor skor
Ringkasan semua skor model akhir (K-Means, K=5). **Metrik internal** (tanpa label) menilai seberapa rapat & terpisah klasternya; **metrik eksternal** membandingkan dengan label generator
dan hanya berfungsi sebagai validasi (di dunia nyata tidak ada). Semua angka dihitung langsung oleh `src/clustering.py: evaluasi_model` — fungsi yang sama dipakai laporan.

In [ ]:
from sklearn.cluster import KMeans

lab = pd.read_csv("data/training/kolektor_label_asli.csv")
ev = C.evaluasi_model(bundle, df, lab)
Zf = bundle["scaler"].transform(C.matriks(df, bundle["fitur"], bundle["kolom_log"]))
stab_seed = float(np.mean([adjusted_rand_score(KMeans(bundle["k"], n_init=50, random_state=42).fit(Zf).labels_,
                                               KMeans(bundle["k"], n_init=50, random_state=s).fit(Zf).labels_) for s in range(1, 8)]))
row_k = tabel_b[tabel_b["k"] == bundle["k"]].iloc[0]
x = ev["eksternal"]

skor = pd.DataFrame([
    ("INTERNAL", "Silhouette (rata-rata)", f"{ev['silhouette']:.3f}", "maks 1; makin tinggi makin baik", ev["tafsir_silhouette"]),
    ("INTERNAL", "Sampel dgn silhouette negatif", f"{ev['pct_silhouette_negatif']:.1f}%", "negatif = kemungkinan salah klaster; makin kecil makin baik", "wajar" if ev["pct_silhouette_negatif"] < 10 else "tinggi"),
    ("INTERNAL", "Davies-Bouldin", f"{ev['davies_bouldin']:.3f}", "makin rendah makin baik (pedoman kasar: <1 baik, 1-2 sedang, >2 kurang)", "sedang" if 1 <= ev["davies_bouldin"] <= 2 else ("baik" if ev["davies_bouldin"] < 1 else "kurang")),
    ("INTERNAL", "Calinski-Harabasz", f"{ev['calinski_harabasz']:.1f}", "makin tinggi makin baik; tak punya skala absolut (dipakai membandingkan K, bagian 3)", "-"),
    ("INTERNAL", "Inertia (WCSS)", f"{ev['inertia']:.1f}", "makin rendah makin baik; otomatis turun saat K naik, tak punya skala absolut", "-"),
    ("INTERNAL", "Jarak dua pusat klaster terdekat", f"{ev['jarak_pusat_terdekat']:.2f}", "satuan z-score; makin besar makin terpisah", "-"),
    ("STABILITAS", "Stabilitas bootstrap (ARI)", f"{row_k['stabilitas_bootstrap']:.3f} ± {row_k['stabilitas_sd']:.3f}", f"ambang yang kita tetapkan: >= {C.STABIL_MIN}", "tepat di ambang" if abs(row_k["stabilitas_bootstrap"] - C.STABIL_MIN) <= 0.03 else ("lolos" if row_k["stabilitas_bootstrap"] >= C.STABIL_MIN else "tidak lolos")),
    ("STABILITAS", "Stabilitas antar seed inisialisasi (ARI)", f"{stab_seed:.3f}", "1 = identik untuk semua seed", "baik" if stab_seed >= 0.9 else "sedang"),
    ("STABILITAS", "BIC GMM", f"{row_k['bic_gmm']:.0f}", "dipakai MEMILIH K (bukan skor mutlak)", "-"),
    ("EKSTERNAL*", "Adjusted Rand Index (ARI)", f"{x['ari']:.3f}", "0 = acak, 1 = identik dgn label generator", "sebagian cocok"),
    ("EKSTERNAL*", "Normalized Mutual Information", f"{x['nmi']:.3f}", "0-1; makin tinggi makin cocok", "sebagian cocok"),
    ("EKSTERNAL*", "Homogenitas", f"{x['homogenitas']:.3f}", "tiap klaster berisi 1 segmen asli?", "sedang"),
    ("EKSTERNAL*", "Kelengkapan", f"{x['kelengkapan']:.3f}", "tiap segmen asli utuh di 1 klaster?", "sedang"),
    ("EKSTERNAL*", "V-measure", f"{x['v_measure']:.3f}", "rata-rata harmonik homogenitas & kelengkapan", "sedang"),
    ("EKSTERNAL*", "Purity", f"{x['purity']:.3f}", "porsi kolektor di klaster yang segmen aslinya dominan", "sedang"),
], columns=["Kelompok", "Metrik", "Skor", "Cara membaca", "Penilaian"])
pd.set_option("display.max_colwidth", 90)
print(f"MODEL: K-Means K={ev['k']} | {ev['n']} kolektor | {ev['n_fitur']} fitur | versi {bundle['versi']}\n")
print(skor.to_string(index=False))
print("\n* eksternal = hanya validasi terhadap label generator (tidak ada di dunia nyata); tidak dipakai memilih K/fitur.")

In [ ]:
# silhouette per segmen + plot silhouette per sampel
ps = pd.DataFrame(ev["per_segmen"]).T.rename(columns={"nama": "segmen", "n": "ukuran", "silhouette": "silhouette", "pct_negatif": "% sampel negatif"})
print(ps.to_string())

ss, sid = ev["sampel_silhouette"], ev["segmen_sampel"]
fig, ax = plt.subplots(figsize=(8, 5)); y = 0
for k in sorted(set(sid)):
    v = np.sort(ss[sid == k]); ax.barh(np.arange(y, y + len(v)), v, height=1.0, label=f"{k}. {bundle['segmen'][k]['nama']}"); y += len(v) + 4
ax.axvline(ev["silhouette"], color="red", ls="--", label=f"rata-rata {ev['silhouette']:.3f}"); ax.axvline(0, color="black", lw=.8)
ax.set_yticks([]); ax.set_xlabel("nilai silhouette per kolektor"); ax.set_title("Silhouette per kolektor, dikelompokkan per segmen"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
terbaik = max(ev["per_segmen"].items(), key=lambda kv: kv[1]["silhouette"]); terlemah = min(ev["per_segmen"].items(), key=lambda kv: kv[1]["silhouette"])
print("KESIMPULAN EVALUASI (dibuat otomatis dari skor di atas)")
print(f"* Silhouette {ev['silhouette']:.3f} -> {ev['tafsir_silhouette']} (skala Kaufman-Rousseeuw: <=0,25 berarti kolektor terdistribusi hampir menerus, bukan kelompok alami yang terpisah).")
print(f"* Segmen terbaik terbentuk: '{terbaik[1]['nama']}' (silhouette {terbaik[1]['silhouette']:.2f}); terlemah: '{terlemah[1]['nama']}' ({terlemah[1]['silhouette']:.2f}, {terlemah[1]['pct_negatif']:.0f}% anggotanya lebih dekat ke klaster lain).")
print(f"* Stabilitas antar seed {stab_seed:.2f} (inisialisasi tidak berpengaruh), tetapi stabilitas bootstrap {row_k['stabilitas_bootstrap']:.2f} +/- {row_k['stabilitas_sd']:.2f} berada di tepi ambang -> jumlah segmen adalah perkiraan.")
print(f"* Terhadap label generator: ARI {x['ari']:.2f}, V-measure {x['v_measure']:.2f}, purity {x['purity']:.2f} -> struktur yang ditanam terbaca SEBAGIAN.")
print("* PENGGUNAAN YANG WAJAR: segmen sebagai RINGKASAN DESKRIPTIF (dashboard, label di UI). TIDAK wajar: menganggapnya kelompok alami yang pasti, atau mengambil keputusan per orang berdasarkan segmen.")

## 9. Mengapa silhouette-nya rendah? (diagnosis)
Silhouette bukan "nilai kerja kita" — ia mengukur **seberapa terpisah kelompok yang memang ada di data**. Jika kolektor tersebar menerus, tidak ada algoritma yang bisa membuatnya tinggi.
Cara membuktikannya: bandingkan dengan **plafon** (silhouette dari *label asli* generator — jawaban yang kita tahu benar) dan **baseline tanpa struktur** (data acak berkovarians sama).

In [ ]:
dg = C.diagnosa_silhouette(bundle, df, lab)
tabel = pd.DataFrame([
    ("Label ASLI generator (plafon: jawaban benar)", dg["label_asli"]),
    (f"Baseline TANPA struktur (Gaussian, K={bundle['k']}, rata-rata 30 ulangan)", dg["baseline_tanpa_struktur"]),
    (f"K-Means kita (K={bundle['k']}, {dg['n_fitur']} fitur)", dg["model"]),
    (f"[ilustrasi] K-Means K=3, {dg['n_fitur']} fitur", dg["k3"]),
    (f"[ilustrasi] K-Means K=2, {dg['n_fitur']} fitur", dg["k2"]),
    (f"[ilustrasi] K-Means K={bundle['k']}, hanya 4 fitur terkuat", dg["model_fitur_kuat"]),
], columns=["pembanding", "silhouette"])
print(tabel.round(3).to_string(index=False))
print(f"\nbaseline tanpa struktur: {dg['baseline_tanpa_struktur']:.3f} +/- {dg['baseline_sd']:.3f} -> model kita {(dg['model'] - dg['baseline_tanpa_struktur']) / dg['baseline_sd']:.1f} simpangan baku di atasnya (struktur ADA, tapi lemah)")
print(f"pembelian per kolektor: median {dg['pembelian_median']:.0f}; {dg['n_kolektor_le3']} dari {dg['n']} kolektor hanya <= 3 pembelian (fitur per orang berisik)")

fig, ax = plt.subplots(figsize=(9, 3.6))
nama = ["label asli\n(plafon)", "tanpa struktur\n(baseline)", f"K-Means K={bundle['k']}\n(model kita)"]
nilai = [dg["label_asli"], dg["baseline_tanpa_struktur"], dg["model"]]
bar = ax.bar(nama, nilai, color=["tab:gray", "tab:red", "tab:blue"])
for r, v in zip(bar, nilai): ax.text(r.get_x() + r.get_width() / 2, v + .005, f"{v:.3f}", ha="center")
ax.axhline(0.25, color="black", ls=":", lw=1); ax.text(2.45, 0.255, "batas 'ada struktur' (K-R)", ha="right", fontsize=8)
ax.set_ylim(0, 0.32); ax.set_ylabel("silhouette"); ax.set_title("Silhouette rendah karena DATA-nya, bukan karena algoritmanya")
plt.tight_layout(); plt.show()

**Kesimpulan diagnosis**
1. **Plafonnya rendah.** Segmen yang ditanam generator *tumpang tindih* di ruang fitur ini (label asli saja hanya ±0,07). Silhouette tinggi memang tidak mungkin dicapai, bukan karena modelnya buruk.
2. **Struktur ada, tapi lemah.** Model kita (0,18) di atas data tanpa struktur (0,14) dengan selisih sekitar 4 simpangan baku — jadi ia menangkap pola nyata, bukan sekadar membelah awan titik.
3. **Datanya berisik per orang.** Median hanya ±6 pembelian per kolektor dan hampir seperempat kolektor ≤ 3 pembelian → fitur seperti "porsi aliran teratas" sangat tidak stabil.
4. **19 fitur membuat jarak "tercampur".** Dengan hanya 4 fitur terkuat silhouette naik (±0,38), tetapi memilih fitur demi menaikkan silhouette itu **sirkular** (seleksi atas metrik yang sama) — hanya ilustrasi, bukan dasar memilih model.
5. **Silhouette tinggi biasanya muncul bila** kelompok memang terpisah jelas (mis. 3 kota yang berjauhan), bukan pada perilaku manusia yang bersifat kontinu. Nilai 0,2–0,3 pada data perilaku nyata itu lazim.

**Konsekuensi untuk presentasi:** katakan dengan jujur bahwa segmen = ringkasan deskriptif atas sebaran yang menerus. Jangan melaporkan K=2/K=3 atau subset fitur hanya karena silhouette-nya lebih tinggi.